# Model Risk Management (MRM) Framework
## Global Housing Price Predictor — SR 11-7 / SS1/23 / MAS Aligned

### Framework Coverage

| Component | SR 11-7 Requirement | Status |
|---|---|---|
| **Model Card** | Documentation, intended use, limitations | ✅ |
| **Out-of-Time Validation** | Independent backtesting, stress tests | ✅ |
| **Sensitivity Analysis** | Tornado chart, what-if analysis | ✅ |
| **SHAP Explainability** | Black-box auditability | ✅ |
| **Calibration Check** | Kupiec POF test, interval coverage | ✅ |
| **Drift Monitor** | PSI feature drift, concept drift | ✅ |
| **OOD Detector** | Out-of-distribution flagging | ✅ |
| **Expert Override** | Audit-trailed human adjustment | ✅ |
| **Prediction Risk Score** | Composite per-prediction risk rating | ✅ |
| **Audit Trail** | Immutable prediction log | ✅ |
| **Regime Detector** | Bubble / crash / normal classification | ✅ |
| **Tail Risk** | VaR / CVaR on forecast paths | ✅ |
| **Model Inventory** | Governance metadata | ✅ |
| **Validation Report** | Full MRM report generator | ✅ |


## 0. Setup & Configuration

In [1]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath(".."))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from sklearn.model_selection import train_test_split

from src.utils.synthetic_data  import generate_synthetic_data
from src.features.engineering  import FeatureEngineer, get_feature_cols
from src.models.ensemble        import HousingEnsemble, compute_metrics
from src.models.forecaster      import LongTermForecaster
from src.shocks.events          import SHOCK_LIBRARY
from src.models.mrm import (
    ModelCard, OutOfTimeValidator, SensitivityAnalyser,
    ShapExplainer, CalibrationChecker, DriftMonitor,
    OODDetector, ExpertOverrideManager, PredictionRiskScorer,
    ModelAuditTrail, RegimeDetector, TailRiskQuantifier,
    ValidationReportGenerator,
)

os.makedirs("../outputs", exist_ok=True)
os.makedirs("../data",    exist_ok=True)
SEED = 42
np.random.seed(SEED)
print("MRM framework loaded successfully")

MRM framework loaded successfully


## 1. Model Card
### The first gate of SR 11-7: every model must have a documented purpose, assumptions, and prohibited uses before deployment.


In [2]:
mc = ModelCard(
    model_id      = "HOUSING-PRED-002",
    version       = "1.0.0",
    owner         = "Quant Research",
    validator     = "Independent Model Validation",
    approver      = "Chief Risk Officer",
    next_review   = "2026-01-01",
    last_validated= "2025-01-01",
)
mc.print_summary()

  MODEL CARD: Global Housing Price Predictor v1.0.0
  ID         : HOUSING-PRED-002
  Owner      : Quant Research
  Validator  : Independent Model Validation
  Created    : 2026-05-04
  Next review: 2026-01-01

  INTENDED USE:
    Indicative market-value estimation for residential property in major cities. Intended for: (1) portfolio valuation scree...

  MATERIALITY: HIGH — predictions may influence investment decisions involving material capital...

  ASSUMPTIONS (9):
    • Training data is representative of the current market segment
    • Historical price-feature relationships persist into the forecast horizon
    • Macro scenario paths are log-normally distributed around trend
    • Property rights and legal title remain stable (no expropriation modelled)

  LIMITATIONS (10):
    ⚠ Synthetic training data: model has not been validated on live transaction data
    ⚠ Long-horizon forecasts (>10yr) are scenario analysis, NOT price predictions
    ⚠ Policy shocks (e.g. nationalisation

In [3]:
# Export model card to JSON for governance records
card_json = mc.to_json("../outputs/model_card.json")
print("Model card saved to ../outputs/model_card.json")
print("\nKey governance fields:")
import json
card = json.loads(card_json)
for k in ["model_id","version","intended_use","materiality","next_review"]:
    print(f"  {k:25s}: {str(card[k])[:80]}")

Model card saved to ../outputs/model_card.json

Key governance fields:
  model_id                 : HOUSING-PRED-002
  version                  : 1.0.0
  intended_use             : Indicative market-value estimation for residential property in major cities. Int
  materiality              : HIGH — predictions may influence investment decisions involving material capital
  next_review              : 2026-01-01


## 2. Train Production Model (Full Dataset)

In [4]:
# Load or generate data
cache = "../data/Shanghai_listings.csv"
if os.path.exists(cache):
    df_all = pd.read_csv(cache)
    print(f"Loaded cached data: {len(df_all):,} rows")
else:
    df_all = generate_synthetic_data("Shanghai", n=8000, seed=SEED)
    df_all.to_csv(cache, index=False)

print(f"Dataset: {df_all.shape} | Years: {df_all['year'].min()}-{df_all['year'].max()}")

# Feature engineering on full set
fe_prod = FeatureEngineer()
y_all   = df_all["unit_price"]
df_eng  = fe_prod.fit_transform(df_all, y_all)
feat_cols = get_feature_cols(df_eng)

X = df_eng[feat_cols].fillna(-999)
X_tr, X_te, y_tr, y_te = train_test_split(X, y_all, test_size=0.20, random_state=SEED)

model = HousingEnsemble(seed=SEED, n_folds=5, tune=False)
model.fit(X_tr, y_tr, feature_cols=feat_cols)

print("\n── Production Model Metrics (random test split) ──")
prod_metrics = model.evaluate(X_te, y_te, label="Production")

Dataset: (8000, 44) | Years: 2015-2024


  Training 5-fold ensemble (5,440 train + 960 calib)...


  Meta weights: XGB=-0.002 LGB-MSE=0.261 LGB-MAE=0.209



  -- OOF Metrics --
  Stacked Ensemble OOF   | MAE=    4,767 | RMSE=    6,453 | R2=0.9603 | MAPE=6.65% | MdAE=    3,486
  Monotonicity: 7 correctors fitted | max violation=50.5%

  Training quantile models + CQR...


  CQR 80% interval | q_hat=3,241 | empirical coverage=80.1% | n_calib=960
  CQR 50% interval | q_hat=1,175 | empirical coverage=50.1% | n_calib=960


  CQR 90% interval | q_hat=4,674 | empirical coverage=90.1% | n_calib=960

── Production Model Metrics (random test split) ──


  Production             | MAE=    9,161 | RMSE=   11,380 | R2=0.8809 | MAPE=17.30% | MdAE=    8,353


## 3. Out-of-Time Validation
### SR 11-7 requires backtesting on temporally separated data — not just a random split.
Training on 2015–2020, testing on 2021–2024 catches look-ahead bias and tests whether
relationships learned in one market cycle hold in another.


In [5]:
oot = OutOfTimeValidator(train_end_year=2020, test_start_year=2021)
df_tr_oot, df_te_oot = oot.split(df_all)

# Re-fit feature engineer on train-only (critical — no future data leakage)
fe_oot   = FeatureEngineer()
y_tr_oot = df_tr_oot["unit_price"]
df_tr_oot_e = fe_oot.fit_transform(df_tr_oot, y_tr_oot)
fc_oot = get_feature_cols(df_tr_oot_e)

# Re-train on out-of-time train split
X_tr_oot = df_tr_oot_e[fc_oot].fillna(-999)
model_oot = HousingEnsemble(seed=SEED, n_folds=5)
model_oot.fit(X_tr_oot, y_tr_oot, feature_cols=fc_oot)

# Out-of-time test
df_te_oot_e = fe_oot.transform(df_te_oot, is_train=False)
fc_shared   = [c for c in fc_oot if c in df_te_oot_e.columns]
X_te_oot    = df_te_oot_e[fc_shared].fillna(-999)
y_te_oot    = df_te_oot["unit_price"]
preds_oot   = model_oot.predict(X_te_oot)

print("\n── Out-of-Time Metrics (2021-2024 held out) ──")
oot_metrics = compute_metrics(y_te_oot.values, preds_oot, "OOT 2021-2024")

# Year-by-year breakdown
year_rows = []
for yr in sorted(df_te_oot["year"].unique()):
    mask = df_te_oot["year"].values == yr
    if mask.sum() < 10: continue
    m = compute_metrics(y_te_oot.values[mask], preds_oot[mask])
    year_rows.append({"Year": yr, "n": int(mask.sum()),
                      "MAE": f"{m['MAE']:,.0f}",
                      "MAPE": f"{m['MAPE']:.2f}%",
                      "R2": f"{m['R2']:.4f}"})
print("\nYear-by-Year Performance (OOT):")
print(pd.DataFrame(year_rows).to_string(index=False))

  Out-of-time split: train 2015–2020 (4,746 rows) | test 2021–2024 (3,254 rows)


  Training 5-fold ensemble (4,035 train + 711 calib)...


  Meta weights: XGB=0.001 LGB-MSE=0.238 LGB-MAE=0.241



  -- OOF Metrics --
  Stacked Ensemble OOF   | MAE=    4,330 | RMSE=    5,930 | R2=0.9614 | MAPE=6.62% | MdAE=    3,058
  Monotonicity: 7 correctors fitted | max violation=50.2%

  Training quantile models + CQR...


  CQR 80% interval | q_hat=3,014 | empirical coverage=80.2% | n_calib=711
  CQR 50% interval | q_hat=1,331 | empirical coverage=50.1% | n_calib=711


  CQR 90% interval | q_hat=4,066 | empirical coverage=90.2% | n_calib=711



── Out-of-Time Metrics (2021-2024 held out) ──
  OOT 2021-2024          | MAE=   11,939 | RMSE=   16,030 | R2=0.7693 | MAPE=13.44% | MdAE=    8,566

Year-by-Year Performance (OOT):
 Year   n    MAE   MAPE     R2
 2021 812 13,387 14.22% 0.7266
 2022 796 11,082 13.00% 0.7916
 2023 808 10,397 12.61% 0.8185
 2024 838 12,838 13.90% 0.7451


In [6]:
# Stress test on known dislocation periods
print("\n── Stress Test on Known Market Dislocations ──")
df_stress_results = oot.stress_test(
    model_oot, fe_oot, df_all, fc_shared,
    stress_periods={
        "Pre-COVID (2019)"  : (2019, 2019),
        "COVID year (2020)" : (2020, 2020),
        "Rate hike (2022)"  : (2022, 2022),
        "Post-peak (2023)"  : (2023, 2023),
        "Recovery (2024)"   : (2024, 2024),
    }
)


── Stress Test on Known Market Dislocations ──



  ── Stress Test Results ──
           Period   n          MAE      MAPE       R2  Status
 Pre-COVID (2019) 765  8141.797379 11.691520 0.894749 ⚠️ Fail
COVID year (2020) 794  8465.267474 11.276205 0.882333 ⚠️ Fail
 Rate hike (2022) 796 11031.771305 12.983016 0.794047 ⚠️ Fail
 Post-peak (2023) 808 10360.549551 12.578789 0.819841 ⚠️ Fail
  Recovery (2024) 838 12869.144416 13.909771 0.744004 ⚠️ Fail


In [7]:
# Plot OOT performance over time
yr_df = pd.DataFrame(year_rows)
yr_df["MAPE_num"] = yr_df["MAPE"].str.replace("%","").astype(float)
yr_df["MAE_num"]  = yr_df["MAE"].str.replace(",","").astype(float)

fig_oot = go.Figure()
fig_oot.add_trace(go.Bar(x=yr_df["Year"], y=yr_df["MAPE_num"],
    name="MAPE (%)", marker_color="#e63946"))
fig_oot.add_hline(y=mc.performance_thresholds["max_mape_pct"],
    line_dash="dash", line_color="orange",
    annotation_text=f"Threshold ({mc.performance_thresholds['max_mape_pct']}%)",
    annotation_position="right")
fig_oot.update_layout(
    title="Out-of-Time MAPE by Year — SR 11-7 Backtesting",
    xaxis_title="Year", yaxis_title="MAPE (%)", height=380)
fig_oot.show()
fig_oot.write_html("../outputs/mrm_01_oot_performance.html")

## 4. Sensitivity Analysis — Tornado Chart
### Shows which features drive the most price variance. Required for SR 11-7 model limitations documentation.
A well-behaved model should be most sensitive to economically meaningful features (area, location, school quality)
and least sensitive to noise features.


In [8]:
sens = SensitivityAnalyser(model, feature_cols=feat_cols, perturbation=0.10)
df_sens = sens.analyse(X_te, top_n=20)

fig_tornado = go.Figure()
fig_tornado.add_trace(go.Bar(
    x=df_sens["impact_up"], y=df_sens["feature"],
    orientation="h", name="Upward shock (+1 std)",
    marker_color="#2d6a4f"))
fig_tornado.add_trace(go.Bar(
    x=df_sens["impact_dn"], y=df_sens["feature"],
    orientation="h", name="Downward shock (-1 std)",
    marker_color="#e63946"))
fig_tornado.update_layout(
    title="Sensitivity Tornado Chart — Price Impact of 1-std Feature Perturbation",
    barmode="overlay",
    xaxis_title="Price Impact (%)", yaxis={"autorange":"reversed"},
    height=650)
fig_tornado.show()
fig_tornado.write_html("../outputs/mrm_02_tornado.html")

print("\nTop 10 most sensitive features:")
print(df_sens[["feature","impact_up","impact_dn","abs_range"]].head(10).to_string(index=False))


Top 10 most sensitive features:
            feature  impact_up  impact_dn  abs_range
    dist_mean_price  12.571662 -13.919384  26.491047
               year   6.579961  -7.338010  13.917971
  dist_median_price   5.873360  -7.958904  13.832264
      decoration_te   4.359308  -4.634191   8.993499
            log_age  -4.198847   4.787610   8.986457
  floor_category_te   2.018278  -3.485012   5.503290
            lpr_5yr  -3.523649   1.673827   5.197476
     orientation_te   1.276023  -3.090117   4.366140
accessibility_score   2.252835  -2.044748   4.297583
     school_quality   1.371466  -1.327171   2.698637


In [9]:
# Single property what-if analysis
sample_row = X_te.iloc[[0]]
df_whatif = sens.single_property_sensitivity(
    sample_row,
    features_to_test=["log_area","log_dist_subway_m","school_quality",
                       "log_age","lpr_5yr","policy_restriction"],
    pct_changes=[-20,-10,-5,0,5,10,20],
)

fig_wif = px.line(
    df_whatif, x="pct_change", y="price_delta_pct",
    color="feature", facet_col="feature", facet_col_wrap=3,
    title="What-If Analysis — How Each Feature Drives Price",
    labels={"pct_change":"Feature Change (%)","price_delta_pct":"Price Change (%)"},
    height=500,
)
fig_wif.update_traces(mode="lines+markers")
fig_wif.show()
fig_wif.write_html("../outputs/mrm_03_whatif.html")

## 5. SHAP Explainability
### SR 11-7 requires models to be explainable. Tree SHAP gives us:
- **Global**: which features matter most across the whole portfolio?
- **Local**: why did the model assign THIS specific price to THIS property?
This is essential for auditors, regulators, and clients who challenge a valuation.


In [10]:
shap_exp = ShapExplainer(model, feat_cols)
shap_exp.fit(X_te, n_background=300)

# Global importance
print("Computing global SHAP importance...")
df_shap_global = shap_exp.explain_global(X_te, n_samples=500)
if len(df_shap_global) > 0:
    print("\nTop 15 features by mean |SHAP|:")
    print(df_shap_global.head(15)[["feature","mean_abs_shap","mean_shap"]].to_string(index=False))

  SHAP explainer fitted on 300 background samples
Computing global SHAP importance...


 26%|=====               | 130/500 [00:11<00:31]       

 28%|======              | 142/500 [00:12<00:30]       

 31%|======              | 154/500 [00:13<00:29]       

 33%|=======             | 167/500 [00:14<00:27]       

...

 91%|==================  | 457/500 [00:37<00:03]       

 94%|=================== | 470/500 [00:38<00:02]       

 96%|=================== | 482/500 [00:39<00:01]       

 99%|===================| 494/500 [00:40<00:00]       


Top 15 features by mean |SHAP|:
            feature  mean_abs_shap  mean_shap
    dist_mean_price       0.224659  -0.008192
               year       0.134515  -0.016522
  dist_median_price       0.120845  -0.005779
            log_age       0.058898  -0.007552
      decoration_te       0.049871  -0.000661
  floor_category_te       0.045640   0.004902
     orientation_te       0.033216  -0.005273
accessibility_score       0.027647  -0.002679
            lpr_5yr       0.025437  -0.000140
             age_sq       0.018122  -0.002647
     dist_std_price       0.017791  -0.000862
     school_quality       0.016930   0.001640
dist_city_center_km       0.010795  -0.001559
  log_dist_subway_m       0.005670   0.000260
    education_score       0.004028   0.000523


In [11]:
# Plot global SHAP
if len(df_shap_global) > 0:
    fig_shap = px.bar(
        df_shap_global.head(20),
        x="mean_abs_shap", y="feature", orientation="h",
        color="mean_shap",
        color_continuous_scale="RdBu",
        color_continuous_midpoint=0,
        title="Global SHAP Feature Importance (mean |SHAP value|)",
        height=600,
    )
    fig_shap.update_layout(yaxis={"autorange":"reversed"})
    fig_shap.show()
    fig_shap.write_html("../outputs/mrm_04_shap_global.html")

In [12]:
# Local explanation for a single property
print("── Local SHAP Explanation: Property #1 ──")
df_local = shap_exp.explain_single(X_te.iloc[[0]])
if len(df_local) > 0:
    # Waterfall-style chart
    df_plot = df_local.head(12).copy()
    colors  = ["#2d6a4f" if v > 0 else "#e63946" for v in df_plot["shap_value"]]
    fig_wf  = go.Figure(go.Bar(
        x=df_plot["shap_value"], y=df_plot["feature"],
        orientation="h", marker_color=colors,
    ))
    fig_wf.update_layout(
        title="Local SHAP Waterfall — Why this price for this property?",
        xaxis_title="SHAP value (log price units)",
        yaxis={"autorange":"reversed"}, height=450,
    )
    fig_wf.show()
    fig_wf.write_html("../outputs/mrm_05_shap_local.html")

── Local SHAP Explanation: Property #1 ──



  Local SHAP Explanation:
  Base value (avg market): 63,861
  Predicted price:         89,849

  Top 10 feature contributions:
    dist_mean_price                    : +0.2105 ↑ increases price
    year                               : +0.1152 ↑ increases price
    dist_median_price                  : +0.1147 ↑ increases price
    log_age                            : -0.0866 ↓ decreases price
    orientation_te                     : -0.0771 ↓ decreases price
    decoration_te                      : -0.0615 ↓ decreases price
    accessibility_score                : +0.0542 ↑ increases price
    floor_category_te                  : +0.0502 ↑ increases price
    dist_city_center_km                : +0.0270 ↑ increases price
    lpr_5yr                            : +0.0258 ↑ increases price


## 6. Prediction Interval Calibration (Kupiec POF Test)
### Are our P10-P90 bands actually containing ~80% of outcomes?
The Kupiec (1995) Proportion-of-Failures test — originally designed for VaR backtesting
under Basel II — tests whether empirical interval coverage matches nominal coverage.
A well-calibrated model should not over- or under-state uncertainty.


In [13]:
calibration = CalibrationChecker()

# Get quantile predictions for test set
unc_te = model.predict_with_uncertainty(X_te)
y_te_arr = y_te.values

# Coverage test
df_calibration = calibration.check_coverage(y_te_arr, unc_te)

# Sharpness
sharpness = calibration.check_sharpness(unc_te)
print(f"\nSharpness metrics (P10-P90 interval width):")
for k, v in sharpness.items():
    print(f"  {k:35s}: {v:,.0f}" if isinstance(v, float) and v > 10
          else f"  {k:35s}: {v:.4f}")


  ── Calibration Check ──
Interval Nominal Cover % Empirical Cover %    Gap Kupiec p-value           Status
  P5-P95             90%             91.9% +1.9pp         0.0100 ⚠️ Miscalibrated
 P10-P90             80%             82.7% +2.7pp         0.0062 ⚠️ Miscalibrated
 P25-P75             50%             47.8% -2.2pp         0.0718     ✅ Calibrated

Sharpness metrics (P10-P90 interval width):
  mean_P10_P90_width                 : 16,224
  median_P10_P90_width               : 14,573
  width_cv                           : 0.3465


In [14]:
# Visualise calibration: actual coverage vs nominal
if len(df_calibration) > 0:
    fig_cal = go.Figure()
    intervals     = df_calibration["Interval"].tolist()
    nominal_vals  = [float(v.replace("%","")) for v in df_calibration["Nominal Cover %"]]
    empirical_vals= [float(v.replace("%","")) for v in df_calibration["Empirical Cover %"]]

    fig_cal.add_trace(go.Bar(name="Nominal", x=intervals, y=nominal_vals,
        marker_color="rgba(69,123,157,0.6)"))
    fig_cal.add_trace(go.Bar(name="Empirical", x=intervals, y=empirical_vals,
        marker_color="#e63946"))
    fig_cal.update_layout(
        title="Quantile Calibration: Nominal vs Empirical Coverage",
        barmode="group", yaxis_title="Coverage (%)",
        height=380,
    )
    fig_cal.show()
    fig_cal.write_html("../outputs/mrm_06_calibration.html")

## 7. Feature & Concept Drift Monitoring
### SR 11-7 requires ongoing performance monitoring. Drift detection answers:
- **Feature drift (PSI)**: Has the input distribution changed since training? (e.g. new property types, different districts)
- **Concept drift**: Is the price-feature relationship itself changing? (e.g. school premium weakening)

PSI thresholds: 🟢 < 0.10 (stable) → 🟡 0.10-0.25 (moderate) → 🟠 0.25-0.40 (significant) → 🔴 > 0.40 (severe)


In [15]:
drift = DriftMonitor(n_bins=10)
drift.fit_reference(df_eng[df_all["year"] <= 2021], feat_cols)

# Simulate recent data (2023-2024) as "production" window
df_recent = df_eng[df_all["year"] >= 2023]
print(f"\nMonitoring drift: reference=2015-2021 | current=2023-2024 ({len(df_recent):,} rows)")
df_psi = drift.monitor(df_recent)

print(f"\nTop 15 features by PSI:")
print(df_psi.head(15).to_string(index=False))

  Drift monitor fitted on 72 numeric features

Monitoring drift: reference=2015-2021 | current=2023-2024 (1,646 rows)
  🔴 4 features with SEVERE drift — model suspension recommended

Top 15 features by PSI:
               Feature    PSI                           Status
               lpr_5yr 5.9139   🔴 Severe drift — SUSPEND MODEL
                  year 5.2454   🔴 Severe drift — SUSPEND MODEL
          lpr_velocity 4.5058   🔴 Severe drift — SUSPEND MODEL
              real_lpr 0.6757   🔴 Severe drift — SUSPEND MODEL
mortgage_payment_proxy 0.3564 🟠 Significant drift — revalidate
       macro_composite 0.1116                 🟡 Moderate drift
     comp_weighted_p25 0.0981                         🟢 Stable
  comp_weighted_median 0.0941                         🟢 Stable
     comp_weighted_p75 0.0586                         🟢 Stable
             longitude 0.0156                         🟢 Stable
                comp_n 0.0138                         🟢 Stable
     log_dist_subway_m 0.0133        

In [16]:
# Concept drift test — simulate performance history over years
print("\n── Concept Drift Test ──")
perf_history = []
for yr in sorted(df_all["year"].unique()):
    subset = df_eng[df_all["year"] == yr]
    if len(subset) < 30: continue
    fc_s = [c for c in feat_cols if c in subset.columns]
    preds_yr = model.predict(subset[fc_s].fillna(-999))
    m = compute_metrics(df_all[df_all["year"] == yr]["unit_price"].values,
                        preds_yr)
    m["year"] = yr
    perf_history.append(m)

concept_result = drift.concept_drift_test(perf_history)

# Plot MAPE over time
perf_df = pd.DataFrame(perf_history)
fig_drift = go.Figure()
fig_drift.add_trace(go.Scatter(x=perf_df["year"], y=perf_df["MAPE"],
    mode="lines+markers", name="MAPE by year",
    line=dict(color="#e63946", width=2)))
fig_drift.add_hline(y=mc.performance_thresholds["max_mape_pct"],
    line_dash="dash", line_color="orange",
    annotation_text="Revalidation threshold")
fig_drift.update_layout(title="Model MAPE Over Time — Concept Drift Monitor",
    xaxis_title="Year", yaxis_title="MAPE (%)", height=380)
fig_drift.show()
fig_drift.write_html("../outputs/mrm_07_drift.html")


── Concept Drift Test ──


  Concept drift: ✅ No significant concept drift


In [17]:
# PSI bar chart
fig_psi = px.bar(
    df_psi.head(20), x="PSI", y="Feature", orientation="h",
    color="PSI",
    color_continuous_scale=["#2d6a4f","#f4a261","#e63946"],
    title="Population Stability Index (PSI) by Feature",
    height=600,
)
fig_psi.add_vline(x=0.10, line_dash="dot", line_color="#f4a261",
    annotation_text="Moderate drift (0.10)")
fig_psi.add_vline(x=0.25, line_dash="dot", line_color="#e63946",
    annotation_text="Significant drift (0.25)")
fig_psi.update_layout(yaxis={"autorange":"reversed"})
fig_psi.show()
fig_psi.write_html("../outputs/mrm_08_psi.html")

## 8. Out-of-Distribution (OOD) Detection
### When a property falls outside the model's training distribution, predictions are less reliable.
Common OOD cases: ultra-luxury properties, unusual building types, new submarkets, foreign investors buying
in unfamiliar zones. Mahalanobis distance flags these automatically.


In [18]:
ood = OODDetector(threshold_pct=97.5)
ood.fit(X_tr.values)

# Score test set
ood_report = ood.report(X_te.values, feat_cols)
n_flagged  = ood_report["ood_flag"].sum()
print(f"OOD detection on test set ({len(X_te):,} properties):")
print(f"  Flagged as OOD: {n_flagged:,} ({n_flagged/len(X_te)*100:.1f}%)")
print(f"  Expected at 97.5th pct threshold: ~2.5%")

# Compare model accuracy on in-distribution vs OOD
ood_mask = ood_report["ood_flag"].values
preds_all = model.predict(X_te)

if ood_mask.sum() > 5:
    mape_id  = np.mean(np.abs((y_te.values[~ood_mask] - preds_all[~ood_mask])
                               / y_te.values[~ood_mask])) * 100
    mape_ood = np.mean(np.abs((y_te.values[ood_mask]  - preds_all[ood_mask])
                               / y_te.values[ood_mask]))  * 100
    print(f"\n  MAPE on in-distribution : {mape_id:.2f}%")
    print(f"  MAPE on OOD flagged     : {mape_ood:.2f}%  ← higher = model less reliable")

  OOD detector fitted | threshold (P98): 4.97
OOD detection on test set (1,600 properties):
  Flagged as OOD: 51 (3.2%)
  Expected at 97.5th pct threshold: ~2.5%



  MAPE on in-distribution : 17.24%
  MAPE on OOD flagged     : 19.15%  ← higher = model less reliable


In [19]:
# OOD score distribution
fig_ood = go.Figure()
fig_ood.add_trace(go.Histogram(
    x=ood_report["ood_score"], nbinsx=50,
    name="OOD Scores", marker_color="#457b9d", opacity=0.75))
fig_ood.add_vline(x=ood._threshold, line_dash="dash", line_color="#e63946",
    annotation_text=f"OOD threshold ({ood._threshold:.1f})",
    annotation_position="top right")
fig_ood.update_layout(
    title="Mahalanobis OOD Score Distribution",
    xaxis_title="OOD Score", yaxis_title="Count", height=380)
fig_ood.show()
fig_ood.write_html("../outputs/mrm_09_ood.html")

## 9. Expert Override System
### SR 11-7 requires override capability with full audit trail.
When an analyst believes the model is wrong (OOD, unique property, policy change),
they can override with a reason code. Overrides > 20% require senior approval.
High override rate = signal to recalibrate the model.


In [20]:
override_mgr = ExpertOverrideManager()

# Example 1: OOD property — analyst applies judgment
sample_pred = float(model.predict(X_te.iloc[[0]])[0])
ov1 = override_mgr.apply_override(
    prediction_id    = "PRED-2024-001",
    model_price      = sample_pred,
    override_price   = sample_pred * 1.08,    # analyst adds 8% for unrenovated premium
    reason_category  = "renovation",
    reason_detail    = "Property recently gut-renovated with premium fit-out, "
                       "not yet reflected in comparable sales",
    analyst_id       = "ANALYST-007",
)

# Example 2: Policy shock override
ov2 = override_mgr.apply_override(
    prediction_id    = "PRED-2024-002",
    model_price      = 95000.0,
    override_price   = 95000.0 * 0.88,       # 12% down for policy risk
    reason_category  = "policy_change",
    reason_detail    = "Property in zone likely affected by upcoming purchase restriction "
                       "policy announcement expected within 30 days",
    analyst_id       = "ANALYST-003",
    approved_by      = "SENIOR-001",
)

# Example 3: Large override (>20%) requires approval
ov3 = override_mgr.apply_override(
    prediction_id    = "PRED-2024-003",
    model_price      = 150000.0,
    override_price   = 150000.0 * 0.70,      # 30% down — unique distressed asset
    reason_category  = "distressed_sale",
    reason_detail    = "Developer receivership — forced liquidation at significant discount",
    analyst_id       = "ANALYST-012",
    approved_by      = "CRO-001",             # requires senior sign-off
)

# Override analytics
print("\n── Override Analytics ──")
analytics = override_mgr.override_analytics()
for k, v in analytics.items():
    if not isinstance(v, dict):
        print(f"  {k:35s}: {v}")

print("\n── Override Log ──")
print(override_mgr.to_dataframe()[
    ["prediction_id","model_price","override_price","override_pct","reason_category","analyst_id"]
].to_string(index=False))

  Override recorded: 84,979 → 91,777 (+8.0%) | renovation
  Override recorded: 95,000 → 83,600 (-12.0%) | policy_change
  Override recorded: 150,000 → 105,000 (-30.0%) | distressed_sale

── Override Analytics ──
  ⚠️  Systematic override bias detected: avg -11.3% → model recalibration needed
  n_overrides                        : 3
  mean_override_pct                  : -11.333333333333334
  median_override_pct                : -12.0
  pct_upward                         : 33.33333333333333
  pct_requiring_approval             : 33.33333333333333
  top_reason                         : renovation

── Override Log ──
prediction_id   model_price  override_price  override_pct reason_category  analyst_id
PRED-2024-001  84979.063572    91777.388658           8.0      renovation ANALYST-007
PRED-2024-002  95000.000000    83600.000000         -12.0   policy_change ANALYST-003
PRED-2024-003 150000.000000   105000.000000         -30.0 distressed_sale ANALYST-012


## 10. Per-Prediction Risk Score
### Every prediction carries a composite risk score that determines whether analyst review is required.
This prevents the model from being used mechanically for high-risk properties
without appropriate human oversight.


In [21]:
risk_scorer = PredictionRiskScorer(
    model_mape=prod_metrics["MAPE"],
    max_acceptable_mape=mc.performance_thresholds["max_mape_pct"],
)

# Score a batch of properties
sample_X   = X_te.iloc[:50]
sample_y   = y_te.iloc[:50]
sample_unc = model.predict_with_uncertainty(sample_X)
sample_ood = ood.score(sample_X.values)

risk_rows = []
for i in range(len(sample_X)):
    score = risk_scorer.score(
        ood_score     = sample_ood[i],
        ood_threshold = ood._threshold,
        q10           = sample_unc["q10"].iloc[i],
        q90           = sample_unc["q90"].iloc[i],
        point_pred    = sample_unc["point_estimate"].iloc[i],
        data_age_days = 30,
    )
    risk_rows.append({
        "property_idx"   : i,
        "predicted_price": round(sample_unc["point_estimate"].iloc[i], 0),
        "risk_score"     : score["risk_score"],
        "risk_rating"    : score["risk_rating"],
        "ci_width_pct"   : score["prediction_interval_pct"],
        "action"         : score["recommended_action"][:50],
    })

df_risk = pd.DataFrame(risk_rows)
print(f"Risk distribution:")
print(df_risk["risk_rating"].value_counts().to_string())
print(f"\nSample predictions with risk scores:")
print(df_risk.head(10).to_string(index=False))

Risk distribution:
risk_rating
🟠 HIGH RISK      48
🟡 MEDIUM RISK     2

Sample predictions with risk scores:
 property_idx  predicted_price  risk_score risk_rating  ci_width_pct                                             action
            0          84979.0        62.4 🟠 HIGH RISK          28.8 Senior analyst review required; consider expert ov
            1         102487.0        52.7 🟠 HIGH RISK          18.7 Senior analyst review required; consider expert ov
            2          95635.0        62.1 🟠 HIGH RISK          28.2 Senior analyst review required; consider expert ov
            3          54298.0        58.0 🟠 HIGH RISK          29.4 Senior analyst review required; consider expert ov
            4          70031.0        66.7 🟠 HIGH RISK          29.4 Senior analyst review required; consider expert ov
            5         121021.0        51.6 🟠 HIGH RISK          13.4 Senior analyst review required; consider expert ov
            6          50952.0        58.2 🟠 HIGH R

In [22]:
# Risk score distribution
fig_risk = px.histogram(df_risk, x="risk_score", color="risk_rating",
    title="Distribution of Prediction Risk Scores",
    labels={"risk_score":"Risk Score (0=low, 100=very high)"},
    color_discrete_map={
        "🟢 LOW RISK":"#2d6a4f",
        "🟡 MEDIUM RISK":"#f4a261",
        "🟠 HIGH RISK":"#e76f51",
        "🔴 VERY HIGH RISK":"#e63946",
    }, height=380)
fig_risk.show()
fig_risk.write_html("../outputs/mrm_10_risk_scores.html")

## 11. Prediction Audit Trail
### Immutable log of all predictions with input hash for tamper detection.
Required for regulatory review and model challenge processes.


In [23]:
audit = ModelAuditTrail()

# Log a batch of predictions
for i in range(min(20, len(sample_X))):
    prop_inputs = sample_X.iloc[i].to_dict()
    risk_info   = risk_scorer.score(
        sample_ood[i], ood._threshold,
        sample_unc["q10"].iloc[i],
        sample_unc["q90"].iloc[i],
        sample_unc["point_estimate"].iloc[i],
    )
    audit.log_prediction(
        property_inputs  = {k: float(v) for k, v in prop_inputs.items()},
        point_prediction = float(sample_unc["point_estimate"].iloc[i]),
        quantile_preds   = {
            "q10": float(sample_unc["q10"].iloc[i]),
            "q90": float(sample_unc["q90"].iloc[i]),
        },
        risk_score    = risk_info,
        model_version = mc.version,
        user_id       = "ANALYST-001",
    )

print("Audit trail (last 10 records):")
print(audit.recent_predictions(10).to_string(index=False))
print(f"\nTotal logged predictions: {len(audit.to_dataframe())}")
print(f"Integrity check: {'✅ Verified' if audit.verify_integrity() else '❌ Tampered'}")

Audit trail (last 10 records):
prediction_id                        timestamp model_version     user_id      inputs_hash  point_prediction     q10      q90  risk_score   risk_rating
     8a1a76f5 2026-05-04T05:55:10.718850+00:00         1.0.0 ANALYST-001 1dd5a7daab0963f8           90702.0 88166.0 109746.0        51.3   🟠 HIGH RISK
     628f5197 2026-05-04T05:55:10.719156+00:00         1.0.0 ANALYST-001 45a8b005b2e0c434           58047.0 42272.0  58888.0        52.5   🟠 HIGH RISK
     471ed09b 2026-05-04T05:55:10.719487+00:00         1.0.0 ANALYST-001 b30d93001c1b7fef           54141.0 42475.0  52000.0        50.3   🟠 HIGH RISK
     26346fe7 2026-05-04T05:55:10.719790+00:00         1.0.0 ANALYST-001 447530048d0f470c           42767.0 23683.0  39817.0        51.5   🟠 HIGH RISK
     bfd43ec2 2026-05-04T05:55:10.720086+00:00         1.0.0 ANALYST-001 dc0725469eedd001           89147.0 89221.0 108473.0        48.2 🟡 MEDIUM RISK
     5b7492ed 2026-05-04T05:55:10.720394+00:00         1.0.0 AN

## 12. Market Regime Detection
### When the market is in a bubble or crash, the model's predictions are less reliable
and uncertainty bands should be widened. The regime detector classifies the current market
and adjusts the recommended use of model outputs accordingly.


In [24]:
regime_detector = RegimeDetector()

print("── Current Market Regime Assessment ──\n")
scenarios_to_test = {
    "Shanghai 2024 (Baseline)": dict(
        yoy_price_growth=3.0, yoy_volume_change=-10.0,
        price_to_income=28, sentiment_score=0.1),
    "Shanghai 2021 (Overheating)": dict(
        yoy_price_growth=18.0, yoy_volume_change=25.0,
        price_to_income=35, sentiment_score=0.75),
    "Hong Kong 2020 (Correction)": dict(
        yoy_price_growth=-8.0, yoy_volume_change=-30.0,
        price_to_income=40, sentiment_score=-0.4),
    "Dubai 2023 (Bubble Risk)": dict(
        yoy_price_growth=22.0, yoy_volume_change=40.0,
        price_to_income=22, sentiment_score=0.85),
    "Berlin 2024 (Correction)": dict(
        yoy_price_growth=-12.0, yoy_volume_change=-25.0,
        price_to_income=18, sentiment_score=-0.5),
    "Tokyo 2024 (Overheating)": dict(
        yoy_price_growth=14.0, yoy_volume_change=15.0,
        price_to_income=20, sentiment_score=0.55),
}

regime_rows = []
for name, params in scenarios_to_test.items():
    result = regime_detector.detect(**params, long_run_avg_growth=5.0)
    regime_rows.append({
        "Market"          : name,
        "Regime"          : result["regime"],
        "YoY Growth %"    : f"{params['yoy_price_growth']:+.0f}%",
        "P/I Ratio"       : params["price_to_income"],
        "Model Reliability": result["model_reliability"].split(" ")[0],
        "Flags"           : ", ".join(result["flags"][:2]) if result["flags"] else "—",
    })

print(pd.DataFrame(regime_rows).to_string(index=False))

── Current Market Regime Assessment ──


  Market Regime: 🟢 NORMAL
  Description  : Price growth aligned with fundamentals
  Reliability  : HIGH — normal operating conditions

  Market Regime: 🔴 BUBBLE
  Description  : Price/fundamental disconnect — high crash risk
  Reliability  : LOW — prices may disconnect from fundamentals
  Flags        : extreme_appreciation, extreme_affordability_stress, euphoric_sentiment

  Market Regime: 🟡 CORRECTION
  Description  : Prices falling — may represent opportunity or further decline
  Reliability  : MODERATE — directional signal reliable, magnitude uncertain
  Flags        : extreme_affordability_stress, price_correction

  Market Regime: 🔴 BUBBLE
  Description  : Price/fundamental disconnect — high crash risk
  Reliability  : LOW — prices may disconnect from fundamentals
  Flags        : extreme_appreciation, euphoric_sentiment

  Market Regime: 🟡 CORRECTION
  Description  : Prices falling — may represent opportunity or further decline
  Reliabil

## 13. Tail Risk — VaR / CVaR on Forecast Paths
### Value-at-Risk and Conditional VaR (Expected Shortfall) quantify the
worst-case loss scenarios at each forecast horizon.
These are essential for investment risk management — how much could I lose
at the 95th percentile over the next 5 years?


In [25]:
# Generate a forecast for a reference property
forecaster = LongTermForecaster(model=model, df_hist=df_all, seed=SEED)
fc_ref = forecaster.forecast(
    district = "\u6d66\u4e1c\u65b0\u533a",
    base_property_params = dict(area_sqm=90, bedrooms=3,
        school_quality=8.5, dist_subway_m=400, dist_city_center_km=8.0),
    horizon  = 30,
    n_sim    = 600,
)

# Current value
current_val = fc_ref[fc_ref["year"] == 2024]["median"].values[0]
print(f"Current property value (2024 median): {current_val:,.0f} CNY/m2")
print(f"Assumed property: 90m2 = {current_val * 90 / 10000:,.0f} wan CNY total\n")

tail_risk = TailRiskQuantifier()
df_var = tail_risk.compute(
    fc_ref, current_val,
    confidence=0.95,
    horizons_yr=[1, 3, 5, 10, 20, 30],
)

Current property value (2024 median): 103,220 CNY/m2
Assumed property: 90m2 = 929 wan CNY total


  ── Tail Risk (VaR / CVaR) | Current Value: 103,220 ──
 Horizon (yr)  Year  Median Price  VaR Price VaR (95%) Loss % CVaR Loss % Upside (95%) %
            1  2025      107161.0    94445.0             8.5%        8.5%         +15.5%
            3  2027      116111.0    82601.0            20.0%       20.0%         +43.9%
            5  2029      126346.0    70537.0            31.7%       31.7%         +77.8%
           10  2034      147723.0    50098.0            51.5%       51.5%        +200.3%
           20  2044      180684.0    21340.0            79.3%       79.3%        +790.3%
           30  2054      226419.0    10322.0            90.0%       90.0%       +2667.4%


In [26]:
# Now compare with shock scenario
fc_shocked = forecaster.forecast(
    district = "\u6d66\u4e1c\u65b0\u533a",
    base_property_params = dict(area_sqm=90, bedrooms=3,
        school_quality=8.5, dist_subway_m=400, dist_city_center_km=8.0),
    horizon  = 30,
    shocks   = [SHOCK_LIBRARY["china_property_debt_crisis"],
                SHOCK_LIBRARY["rate_shock_200bps"]],
    n_sim    = 600,
)

print("\n── Tail Risk Under Shock Scenario ──")
df_var_shocked = tail_risk.compute(fc_shocked, current_val,
    confidence=0.95, horizons_yr=[1, 3, 5, 10, 20, 30])


── Tail Risk Under Shock Scenario ──

  ── Tail Risk (VaR / CVaR) | Current Value: 103,220 ──
 Horizon (yr)  Year  Median Price  VaR Price VaR (95%) Loss % CVaR Loss % Upside (95%) %
            1  2025       76823.0    57299.0            44.5%       44.5%         +-2.3%
            3  2027      109195.0    76103.0            26.3%       26.3%         +35.8%
            5  2029      123494.0    67576.0            34.5%       34.5%         +70.5%
           10  2034      144061.0    49211.0            52.3%       52.3%        +191.0%
           20  2044      175957.0    20522.0            80.1%       80.1%        +756.9%
           30  2054      219145.0    10118.0            90.2%       90.2%       +2625.4%


In [27]:
# VaR comparison plot
fig_var = go.Figure()
# Baseline median & P5
fig_var.add_trace(go.Scatter(x=fc_ref["year"], y=fc_ref["median"],
    name="Baseline Median", line=dict(color="#457b9d", width=2)))
fig_var.add_trace(go.Scatter(x=fc_ref["year"], y=fc_ref["p5"],
    name="Baseline P5 (VaR proxy)", line=dict(color="#457b9d", width=1.5, dash="dash")))
# Shocked
fig_var.add_trace(go.Scatter(x=fc_shocked["year"], y=fc_shocked["median"],
    name="Shocked Median", line=dict(color="#e63946", width=2)))
fig_var.add_trace(go.Scatter(x=fc_shocked["year"], y=fc_shocked["p5"],
    name="Shocked P5 (VaR proxy)", line=dict(color="#e63946", width=1.5, dash="dash")))
fig_var.add_hline(y=current_val, line_dash="dot", line_color="grey",
    annotation_text="Current value")
fig_var.update_layout(
    title="Tail Risk: Baseline vs Shock Scenario (Median + P5)",
    xaxis_title="Year", yaxis_title="Unit Price (CNY/m2)",
    height=460, hovermode="x unified")
fig_var.show()
fig_var.write_html("../outputs/mrm_11_tail_risk.html")

## 14. Full MRM Validation Report

In [28]:
rpt = ValidationReportGenerator(mc)

# Add all validation results to the report
rpt.add_section(
    "Production Model Performance",
    {k: (f"{v:.4f}" if isinstance(v, float) else v)
     for k, v in prod_metrics.items()},
    status=("✅ PASS" if prod_metrics["MAPE"] < mc.performance_thresholds["max_mape_pct"]
            else "⚠️ FAIL"),
)
rpt.add_section(
    "Out-of-Time Validation (2021-2024)",
    {k: (f"{v:.4f}" if isinstance(v, float) else v)
     for k, v in oot_metrics.items()},
    status=("✅ PASS" if oot_metrics["MAPE"] < mc.performance_thresholds["max_mape_pct"] * 1.5
            else "⚠️ FAIL — exceeds OOT threshold"),
)
rpt.add_section(
    "Calibration (Kupiec POF Test)",
    df_calibration.to_string(index=False) if len(df_calibration) > 0 else "N/A",
)
rpt.add_section(
    "Feature Drift (PSI)",
    df_psi.head(10).to_string(index=False),
    status="🟢 No severe drift" if (df_psi["PSI"] > 0.40).sum() == 0 else "🔴 Severe drift",
)
rpt.add_section(
    "Concept Drift",
    concept_result,
)
rpt.add_section(
    "OOD Analysis",
    {
        "OOD flagged (%)": f"{n_flagged/len(X_te)*100:.1f}%",
        "Expected at threshold": "~2.5%",
        "MAPE on in-distribution": f"{mape_id:.2f}%",
        "MAPE on OOD flagged"    : f"{mape_ood:.2f}%",
    },
)
rpt.add_section(
    "Override Analytics",
    {k: v for k, v in analytics.items() if not isinstance(v, dict)},
)
rpt.add_section(
    "Sensitivity — Top 5 Drivers",
    df_sens[["feature","abs_range"]].head(5).to_string(index=False),
)
rpt.add_section(
    "Known Model Limitations",
    mc.limitations,
)

# Generate and save
report_text = rpt.generate("../outputs/mrm_validation_report.txt")
print(report_text[:3000])
print("\n... [full report saved to ../outputs/mrm_validation_report.txt]")

  Validation report saved: ../outputs/mrm_validation_report.txt
  MODEL VALIDATION REPORT
  Global Housing Price Predictor | v1.0.0 | HOUSING-PRED-002
  Generated: 2026-05-04 05:55 UTC

SECTION 1: MODEL IDENTITY & GOVERNANCE
----------------------------------------
  Model ID         : HOUSING-PRED-002
  Owner            : Quant Research
  Validator        : Independent Model Validation
  Approver         : Chief Risk Officer
  Last validated   : 2025-01-01
  Next review      : 2026-01-01
  Materiality      : HIGH — predictions may influence investment decisions involving material capital

SECTION 2: INTENDED USE & PROHIBITIONS
----------------------------------------
  Intended use: Indicative market-value estimation for residential property in major cities. Intended for: (1) portfolio valuation scree
  Prohibited uses (6):
    ✗ Regulatory capital / RWA calculations
    ✗ IFRS 13 / ASC 820 fair value for financial reporting
    ✗ Mortgage collateral valuation (appraisal)
    ✗ REIT N

## 15. MRM Summary Dashboard

In [29]:
fig_dash = go.Figure()

metrics_data = {
    "Metric"   : ["MAPE (%)", "R²", "OOT MAPE (%)", "OOD Rate (%)",
                  "P10-P90 Coverage (%)", "Concept Drift p-val"],
    "Value"    : [
        round(prod_metrics["MAPE"], 2),
        round(prod_metrics["R2"], 4),
        round(oot_metrics["MAPE"], 2),
        round(n_flagged / len(X_te) * 100, 1),
        float(df_calibration["Empirical Cover %"].iloc[1].replace("%",""))
            if len(df_calibration) > 1 else 80.0,
        round(concept_result.get("p_value", 1.0), 4),
    ],
    "Threshold": [
        mc.performance_thresholds["max_mape_pct"],
        mc.performance_thresholds["min_r2"],
        mc.performance_thresholds["max_mape_pct"] * 1.5,
        5.0,
        mc.performance_thresholds["min_quantile_coverage_80pct"],
        0.05,
    ],
    "Status"   : [],
}

# Higher-is-better for R2 and coverage; lower-is-better for rest
higher_better = {"R²", "P10-P90 Coverage (%)"}
lower_better_pval = {"Concept Drift p-val"}
for i, metric in enumerate(metrics_data["Metric"]):
    val  = metrics_data["Value"][i]
    thr  = metrics_data["Threshold"][i]
    if metric in higher_better:
        status = "✅" if val >= thr else "⚠️"
    elif metric in lower_better_pval:
        status = "✅" if val >= thr else "⚠️ Trend detected"
    else:
        status = "✅" if val <= thr else "⚠️"
    metrics_data["Status"].append(status)

df_dash = pd.DataFrame(metrics_data)
print("=" * 60)
print("  MRM DASHBOARD SUMMARY")
print("=" * 60)
print(df_dash.to_string(index=False))
print("=" * 60)

all_pass = all(s == "✅" for s in df_dash["Status"])
if all_pass:
    print("\n  ✅ MODEL STATUS: ALL CHECKS PASSED — approved for intended use")
else:
    n_warn = sum(1 for s in df_dash["Status"] if "⚠️" in s)
    print(f"\n  ⚠️  MODEL STATUS: {n_warn} WARNING(S) — review before use")

  MRM DASHBOARD SUMMARY
              Metric   Value  Threshold            Status
            MAPE (%) 17.3000       8.00                ⚠️
                  R²  0.8809       0.92                ⚠️
        OOT MAPE (%) 13.4400      12.00                ⚠️
        OOD Rate (%)  3.2000       5.00                 ✅
P10-P90 Coverage (%) 82.7000      72.00                 ✅
 Concept Drift p-val  0.0073       0.05 ⚠️ Trend detected

  ⚠️  MODEL STATUS: 4 WARNING(S) — review before use


## 16. SR 11-7 Compliance Checklist

| Requirement | Implementation | Status |
|---|---|---|
| **Model purpose documented** | ModelCard.intended_use | ✅ |
| **Prohibited uses listed** | ModelCard.prohibited_uses | ✅ |
| **Assumptions documented** | ModelCard.assumptions | ✅ |
| **Limitations disclosed** | ModelCard.limitations | ✅ |
| **Out-of-time backtesting** | OutOfTimeValidator | ✅ |
| **Stress testing** | OutOfTimeValidator.stress_test | ✅ |
| **Sensitivity analysis** | SensitivityAnalyser | ✅ |
| **Explainability (SHAP)** | ShapExplainer | ✅ |
| **Interval calibration** | CalibrationChecker + Kupiec | ✅ |
| **Feature drift (PSI)** | DriftMonitor | ✅ |
| **Concept drift test** | DriftMonitor.concept_drift_test | ✅ |
| **OOD detection** | OODDetector (Mahalanobis) | ✅ |
| **Expert override** | ExpertOverrideManager + audit | ✅ |
| **Per-prediction risk** | PredictionRiskScorer | ✅ |
| **Audit trail** | ModelAuditTrail | ✅ |
| **Regime detection** | RegimeDetector | ✅ |
| **VaR / CVaR** | TailRiskQuantifier | ✅ |
| **Validation report** | ValidationReportGenerator | ✅ |
| **Governance metadata** | ModelCard + ModelInventoryEntry | ✅ |
| **Performance thresholds** | ModelCard.performance_thresholds | ✅ |
| **Revalidation schedule** | ModelCard.next_review | ✅ |
| **Retirement criteria** | ModelInventoryEntry.retirement_criteria | ✅ |

### Residual Risks (Not Fully Mitigated)
| Risk | Mitigation Status | Residual |
|---|---|---|
| **Synthetic training data** | Calibrated distributions | HIGH — awaits live data |
| **Long-horizon forecast** | CI bands + scenario analysis | MEDIUM — fundamental limit |
| **Regime structural breaks** | RegimeDetector + shock system | MEDIUM — non-parametric |
| **Regulatory change** | Shock library (policy shocks) | MEDIUM — expert judgment required |
| **Climate physical risk** | Flood/earthquake flags | HIGH — needs specialist model |
| **Cross-market contagion** | Correlated macro scenarios | MEDIUM — simplified |
